# 40_serving_export — Serving, Optimization, Reconciliation

Final step in the pipeline:
1. **Reconciliation** — counts + orphan checks (warns, does not fail)
2. **OPTIMIZE** — Liquid Clustering for fact, dim_user, dim_track
3. **Statistics** — `ANALYZE TABLE` for query optimization
4. **Serving export** — if no SQL warehouse, write Gold as CSV/Parquet to `serving/`

In [ ]:
%run ./00_config


## Step 1: Reconciliation

In [ ]:
from datetime import datetime, timezone

_t0 = datetime.now(timezone.utc)
recon_ok = True
recon_notes = []

try:
    # Row count check: fact vs silver.events
    silver_n = spark.table(SILVER_EVENTS).count()
    fact_n   = spark.table(GOLD['fact_listening']).count()
    if silver_n != fact_n:
        msg = f'ROW COUNT MISMATCH: silver={silver_n:,} vs fact={fact_n:,}'
        recon_notes.append(msg)
        recon_ok = False
        print(f'  WARNING: {msg}')
    else:
        print(f'  Row counts match: {fact_n:,}')

    # Orphan FK check — device
    orphan_dev = spark.sql(f"""
        SELECT COUNT(*) AS n FROM {table_ref(GOLD['fact_listening'])} f
        WHERE NOT EXISTS (SELECT 1 FROM {table_ref(GOLD['dim_device'])} d WHERE d.device_sk=f.device_sk)
    """).first()['n']
    if orphan_dev > 0:
        recon_notes.append(f'Orphan device FKs: {orphan_dev}')
    print(f'  Orphan device FKs: {orphan_dev}')

    # Orphan FK check — user
    orphan_usr = spark.sql(f"""
        SELECT COUNT(*) AS n FROM {table_ref(GOLD['fact_listening'])} f
        WHERE NOT EXISTS (SELECT 1 FROM {table_ref(GOLD['dim_user'])} u WHERE u.user_sk=f.user_sk)
    """).first()['n']
    if orphan_usr > 0:
        recon_notes.append(f'Orphan user FKs: {orphan_usr}')
    print(f'  Orphan user FKs: {orphan_usr}')

    # Quarantine rate
    q_rate_row = spark.sql(f"""
        SELECT SUM(rejection_count)*1.0/NULLIF(SUM(total_count),0) AS rate
        FROM {table_ref(GOLD['audit_quality_summary'])} WHERE layer='bronze'
    """).first()
    q_rate = q_rate_row['rate'] or 0.0
    if q_rate > QUARANTINE_ALERT_RATE:
        recon_notes.append(f'Quarantine rate {q_rate:.2%} exceeds {QUARANTINE_ALERT_RATE:.0%} alert threshold')
    print(f'  Overall quarantine rate: {q_rate:.2%}')

    log_run('SILVER_TO_GOLD','reconciliation',f'silver={silver_n},fact={fact_n}',
            _t0,'SUCCESS' if recon_ok else 'FAILURE',
            rows_read=silver_n, rows_inserted=fact_n,
            error_message='; '.join(recon_notes) if recon_notes else None)
    print('\n  Reconciliation done.')

except Exception as _e:
    print(f'Reconciliation error (non-fatal): {_e}')
    log_run('SILVER_TO_GOLD','reconciliation',START_DATE,_t0,'FAILURE',error_message=str(_e))


## Step 2: OPTIMIZE + Liquid Clustering (with before/after evidence)

In [ ]:
_t0 = datetime.now(timezone.utc)

def capture_detail(tref):
    """Return (numFiles, sizeInBytes) from DESCRIBE DETAIL."""
    row = spark.sql(f'DESCRIBE DETAIL {table_ref(tref)}').first()
    return row['numFiles'], row['sizeInBytes']

to_optimize = [
    GOLD['fact_listening'],
    GOLD['dim_user'],
    GOLD['dim_track'],
    BRONZE_EVENTS,   # occasional: many small daily files
]

for tref in to_optimize:
    try:
        files_before, bytes_before = capture_detail(tref)
        print(f'\nOPTIMIZE {tref}')
        print(f'  Before: {files_before} files, {bytes_before/1024**2:.1f} MB')
        opt_result = spark.sql(f'OPTIMIZE {table_ref(tref)}')
        files_after, bytes_after = capture_detail(tref)
        print(f'  After:  {files_after} files, {bytes_after/1024**2:.1f} MB')
        print(f'  Files reduced: {files_before - files_after:+d}')
    except Exception as _e:
        print(f'  OPTIMIZE failed for {tref}: {_e}')


## Step 3: Compute Statistics

In [ ]:
for tref in [GOLD['fact_listening'], GOLD['dim_user'], GOLD['dim_track']]:
    try:
        print(f'ANALYZE {tref} ...')
        spark.sql(f'ANALYZE TABLE {table_ref(tref)} COMPUTE STATISTICS FOR ALL COLUMNS')
        print(f'  Done.')
    except Exception as _e:
        print(f'  Analyze failed (non-fatal): {_e}')


## Step 4: Serving Export (fallback if no SQL warehouse)

In [ ]:
# Export Gold tables as Parquet to serving/ for Power BI import mode fallback
# In a workspace with a SQL warehouse, Power BI connects directly — skip this.
SERVING_ROOT = f'{LANDING_ROOT}/serving'

SERVING_TABLES = [
    GOLD['fact_listening'],
    GOLD['dim_user'],
    GOLD['dim_track'],
    GOLD['dim_date'],
    GOLD['dim_time'],
    GOLD['dim_device'],
    GOLD['peak_usage_patterns'],
    GOLD['subscription_propensity_segments'],
    GOLD['content_performance_trends'],
    GOLD['wrapped_user_year'],
    GOLD['audit_pipeline_runs'],
    GOLD['audit_quality_summary'],
    GOLD['audit_freshness'],
]

for tref in SERVING_TABLES:
    tname = tref.split('.')[-1]
    out_path = f'{SERVING_ROOT}/{tname}'
    try:
        spark.table(tref).write.format('parquet').mode('overwrite').save(out_path)
        print(f'  Exported: {tname} -> {out_path}')
    except Exception as _e:
        print(f'  Export failed for {tname}: {_e}')

print(f'\nServing export complete: {SERVING_ROOT}')


## Proof: Final Pipeline Summary

In [ ]:
print('=== Final Pipeline Execution Log (SILVER_TO_GOLD) ===')
display(spark.sql(f"""
    SELECT target_table, status, rows_read, rows_inserted, rows_updated,
           rows_deleted, ROUND(duration_sec,1) AS duration_sec, error_message
    FROM {table_ref(EXECUTION_LOG)}
    WHERE layer='SILVER_TO_GOLD' AND batch_id='{BATCH_ID}'
    ORDER BY start_time
"""))

print('\n=== Gold Table Freshness ===')
display(spark.table(GOLD['audit_freshness']))
